# RankDSL Result Inspection

This notebook reads persisted per-request JSON files under `results/saved_rankings` and provides:

- Pareto plots between relevance and constraint satisfaction
- Compile-failure case inspection
- DSL consistency heatmaps across paraphrases


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

CACHE_DIR = Path('../results/saved_rankings')
CACHE_DIR = CACHE_DIR.resolve()
files = sorted(CACHE_DIR.glob('*.json'))
print('cache_dir =', CACHE_DIR)
print('files =', len(files))

In [ ]:
rows = []
raw_results = []
for path in files:
    payload = json.loads(path.read_text(encoding='utf-8'))
    raw_results.append(payload)
    for run_index, run in enumerate(payload.get('rankdsl', {}).get('runs', [])):
        ilp = run.get('ilp', {})
        rows.append({
            'request_id': payload.get('request_id'),
            'scenario_id': payload.get('scenario_id'),
            'user_id': payload.get('user_id'),
            'run_index': run_index,
            'compile_success': 'compile_error' not in run,
            'compile_error': json.dumps(run.get('compile_error', []), ensure_ascii=False),
            'fallback_used': bool(run.get('fallback_used') or ilp.get('fallback_used')),
            'ndcg@10': ilp.get('ndcg@10', 0.0),
            'hit@10': ilp.get('hit@10', 0.0),
            'constraint_satisfaction': ilp.get('constraint_satisfaction', 0.0),
            'filter_ok': ilp.get('filter_ok', 0.0),
            'quota_satisfaction': ilp.get('quota_satisfaction', 0.0),
            'diversity_satisfaction': ilp.get('diversity_satisfaction', 0.0),
            'sliding_window_ok': ilp.get('sliding_window_ok', 0.0),
            'sliding_violation_rate': ilp.get('sliding_violation_rate', 0.0),
            'ild_score': ilp.get('ild_score', 0.0),
            'compiled_dsl_canonical': run.get('compiled_dsl_canonical'),
        })
df = pd.DataFrame(rows)
df.head()

## Pareto Plot

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=df,
    x='ndcg@10',
    y='constraint_satisfaction',
    hue='scenario_id',
    style='compile_success',
    alpha=0.7,
)
plt.title('Pareto Frontier: Relevance vs Constraint Satisfaction')
plt.tight_layout()
plt.show()

## Compile Failure Analysis

In [ ]:
failures = df.loc[~df['compile_success']].copy()
failures.groupby('scenario_id').size().sort_values(ascending=False)

In [ ]:
failure_examples = failures[['request_id', 'scenario_id', 'compile_error']].drop_duplicates().head(10)
failure_examples

## DSL Consistency Heatmap

In [ ]:
consistency_rows = []
for payload in raw_results:
    runs = payload.get('rankdsl', {}).get('runs', [])
    canonical = [run.get('compiled_dsl_canonical') for run in runs]
    for i, left in enumerate(canonical):
        for j, right in enumerate(canonical):
            consistency_rows.append({
                'request_id': payload.get('request_id'),
                'left': i,
                'right': j,
                'equal': float(left is not None and left == right),
            })
consistency_df = pd.DataFrame(consistency_rows)
heatmap_data = consistency_df.groupby(['left', 'right'])['equal'].mean().unstack(fill_value=0.0)
plt.figure(figsize=(5, 4))
sns.heatmap(heatmap_data, annot=True, vmin=0.0, vmax=1.0, cmap='Blues')
plt.title('DSL Consistency Across Paraphrases')
plt.tight_layout()
plt.show()